## Step 1: Import the required libraries

In [5]:
!pip install statsmodels
!pip install sklearn
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set

ModuleNotFoundError: No module named 'sklearn'

## Step 2: Load and inspect the dataset

In [ ]:
csv_path = Path('grunfeld_investment_nonseasonal.csv')
if not csv_path.exists():
    raise FileNotFoundError("Place grunfeld_investment_nonseasonal.csv in the same folder as this notebook.")

df = pd.read_csv(csv_path)
print('Full dataset shape:', df.shape)
display(df.head())
print('Missing values by column:')
print(df.isnull().sum())
print('Firms:', sorted(df['Firm'].unique()))

## Step 3: Select one company and prepare the dates

The file contains several companies. We select General Motors so the model follows one company's values over time instead of mixing different companies into one time series. The data is annual, so it does not have monthly or weekly seasonal cycles.

In [ ]:
company_df = df[df['Firm'] == 'General Motors'].copy()
company_df['Date'] = pd.to_datetime(company_df['Year'].astype(str) + '-01-01')
company_df = company_df.sort_values('Date').set_index('Date')
company_df = company_df.asfreq('YS')

print('Company selected: General Motors')
print('Number of annual observations:', len(company_df))
display(company_df.head())
print('Missing values after selecting the company:')
print(company_df[['Investment', 'Firm_Value', 'Capital_Stock']].isnull().sum())

## Step 4: Visualize annual investment

The graph helps us see how General Motors' investment changed over the years.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(company_df.index, company_df['Investment'], marker='o')
plt.title('General Motors — Annual Investment')
plt.xlabel('Year')
plt.ylabel('Investment (1947 dollars)')
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()

## Step 5: Select the target and explanatory variables

- **Target (`Investment`):** what we want to predict.
- **Explanatory variables (`Firm_Value`, `Capital_Stock`):** extra business information supplied to the model.

We do not use the `Firm` name as a predictor because we already filtered the data to one firm.

In [ ]:
target = company_df['Investment']
exog = company_df[['Firm_Value', 'Capital_Stock']]

# Use the first 16 years for training and the last 4 years for testing.
TEST_SIZE = 4
train_target = target.iloc[:-TEST_SIZE]
test_target = target.iloc[-TEST_SIZE:]
train_exog = exog.iloc[:-TEST_SIZE]
test_exog = exog.iloc[-TEST_SIZE:]

print('Training observations:', len(train_target))
print('Testing observations:', len(test_target))
print('Test years:', test_target.index.year.tolist())

## Step 6: Build and train the non-seasonal ARIMAX model

The `ARIMA` class accepts `exog` variables. When these extra variables are supplied, this use is commonly called **ARIMAX**.

`order=(1, 1, 0)` means one autoregressive lag, first differencing, and no moving-average error lag. These are simple baseline settings for this small annual dataset, not guaranteed best settings.

In [ ]:
model = ARIMA(
    train_target,
    exog=train_exog,
    order=(1, 1, 0)
)
result = model.fit()
print(result.summary())

## Step 7: Forecast the final four years

The explanatory variables from the test period are supplied so we can compare the forecast with actual investment. For a real future forecast, future values of these explanatory variables would need to be known or forecast separately.

In [ ]:
forecast = result.forecast(steps=len(test_target), exog=test_exog)
comparison = pd.DataFrame({
    'Actual_Investment': test_target,
    'Predicted_Investment': forecast
})
display(comparison)

## Step 8: Evaluate using MAE and RMSE

In [ ]:
mae = mean_absolute_error(test_target, forecast)
rmse = np.sqrt(mean_squared_error(test_target, forecast))
print(f'MAE:  {mae:.3f}')
print(f'RMSE: {rmse:.3f}')

## Step 9: Plot actual versus predicted investment

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train_target.index, train_target, label='Training Investment')
plt.plot(test_target.index, test_target, marker='o', label='Actual Test Investment')
plt.plot(forecast.index, forecast, marker='x', linestyle='--', label='ARIMAX Forecast')
plt.title('General Motors — Actual vs Predicted Annual Investment')
plt.xlabel('Year')
plt.ylabel('Investment (1947 dollars)')
plt.legend()
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()